First is the **set-up cell** - run it first.

* Qiskit has changed a lot over the years, and some older commands (`Aer`, `BasicAer`, `execute`,
  `IBMQ`) no longer exist. The helper file `qiskit_backend_config.py` (in this folder) provides
  stand-ins for them, so the older code below still works.
* `sys.path.insert(...)` tells Python to look for that helper file in the current folder.
* `BACKEND_NAME = "qasm_simulator"` chooses **where the circuits run**. A *backend* is either a
  simulator on your own computer or a real quantum computer at IBM. The `qasm_simulator` imitates
  a real device: it *measures* the qubits many times ("shots") and counts the results.
* If you have an IBM Quantum account you could set a token and pick a real device, but nothing in
  this notebook needs that.
* The last line prints which backend was created, so you know the cell worked.

In [1]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


Backend ready: LegacyAerSimulator('aer_simulator')


This cell tries to **log in to IBM Quantum**. Without a saved IBM token it just warns and the
notebook keeps using the local simulator; nothing here needs a real quantum computer.

In [2]:
from qiskit_backend_config import IBMQ
# Loading your IBM Q account(s)
IBMQ.load_account()

**import** the tools we need: `numpy` for arrays, `matplotlib` for plots, Qiskit's
circuit classes, the transpiler helper, `SparsePauliOp` (observables), `BitArray` (the format in which measurement results come back) and, most importantly, the **`StatevectorSampler`**, the
*Sampler primitive*, which we explore in this notebook.

In [3]:
# Useful additional packages
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from math import pi
# Importing standard Qiskit libraries and configuring account
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray
from qiskit.primitives import StatevectorEstimator


from qiskit.primitives import StatevectorSampler

Next code cell **builds a 10-qubit GHZ circuit**.

* A **GHZ state** is $\frac{1}{\sqrt 2}\big(|00\ldots0\rangle + |11\ldots1\rangle\big)$: all qubits
  are 0, or all qubits are 1, each with probability 50 %. The qubits are perfectly correlated.
* `h(0)` puts qubit 0 into an equal superposition of 0 and 1.
* `cx(range(0, 9), range(1, 10))` applies CNOTs 0→1, 1→2, …, 8→9. Each CNOT copies the value of the
  previous qubit onto the next, spreading the superposition along the chain.
* `measure_all()` adds a measurement to every qubit and creates a classical register called
  **`meas`** to store the 10 results.

In [4]:
# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

This cell **transpiles** the circuit: it rewrites the circuit into gates and qubit connections
that the target can run, with light optimisation (`optimization_level=1`). The `pm` object is
reused later in the notebook.

In [5]:

# transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1)
transpiled_circuit = pm.run(circuit)

This cell **runs the circuit with the Sampler**.

* The **Sampler** is the second Qiskit primitive. Unlike the Estimator (which returns averages),
  it returns the **raw measurement outcomes**: which bitstring came out in each run ("shot").
* `StatevectorSampler()` simulates on your computer; by default it takes **1024 shots**.
* `sampler.run([...])` submits the job, `.result()` waits for the answer, and `result[0].data` is
  the data of our (only) circuit.

In [6]:
sampler = StatevectorSampler()

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data

This cell **prints the DataBin**. It contains one entry, `meas`, named after the classical
register that `measure_all()` created.

In [7]:
print(f"Databin: {data}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=10>))



This cell **takes out the measurement results** as a `BitArray`. A `BitArray` stores, for every
shot, the bits that were measured, in a compact form. The printout tells us the number of shots
(1024) and bits per shot (10).

In [8]:
# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")

BitArray: BitArray(<shape=(), num_shots=1024, num_bits=10>)



Next code cell **prints the shape of the stored data**: `(1024, 2)`.

* 1024 rows = one per shot.
* 2 columns because the 10 bits of each shot are packed into **bytes** (1 byte = 8 bits), and 10
  bits need 2 bytes.

In [9]:
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")

The shape of register `meas` is (1024, 2).



This cell **prints the raw bytes, shot by shot**. (The label says `alpha`, but this is the register
`meas`.)

* `[0 0]` means all 10 bits are 0, i.e. the outcome `0000000000`.
* `[3 255]` means all 10 bits are 1: $255 = 11111111_2$ holds eight 1s and $3 = 11_2$ holds two.
* So **only two outcomes ever appear**, all zeros or all ones, exactly as expected for a GHZ state.

In [10]:
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

The bytes in register `alpha`, shot by shot:
[[  3 255]
 [  0   0]
 [  0   0]
 ...
 [  0   0]
 [  3 255]
 [  3 255]]



This cell **converts the results into a more readable form**: a dictionary of *counts*.

* `get_counts()` returns `{bitstring: number of times it occurred}`.
* Expect two entries, `'0000000000'` and `'1111111111'`, each roughly 512 times (≈ 50 %). They are
  not exactly 512 because measurement outcomes are random, like tossing a coin 1024 times.

In [11]:
#It can sometimes be convenient to convert from the bytes format in the 
# BitArray to bitstrings. 
# The get_count method returns a dictionary mapping bitstrings to the number of times 
# that they occurred.

# optionally convert the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'1111111111': 507, '0000000000': 517}


When a circuit contains more than one classical register, the results are stored in different BitArray objects. The following example modifies the previous snippet by splitting the classical register into two distinct registers:


Next code cell **builds the same GHZ circuit again, but stores the results in two separate classical
registers**:

* `alpha` (1 bit) gets the result of qubit 0;
* `beta` (9 bits) gets the results of qubits 1–9.

`qreg := QuantumRegister(10)` uses Python's *walrus operator* `:=`, which creates the register
and gives it a name in one step. `circuit.measure([0], alpha)` measures qubit 0 into `alpha`;
`circuit.measure(range(1, 10), beta)` measures qubits 1–9 into `beta`.

In [12]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

This cell **transpiles and runs the new circuit**, then prints the two `BitArray`s. With two
registers, the results are stored separately and can be accessed by name: `data.alpha` and
`data.beta` (1 bit and 9 bits per shot, 1024 shots each).

In [13]:
# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=1024, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=1024, num_bits=9>)


Leveraging BitArray objects for performant post-processing
Since arrays generally offer better performance compared to dictionaries, it is advisable to perform any post-processing directly on the BitArray objects rather than on dictionaries of counts. The BitArray class offers a range of methods to perform some common post-processing operations:


This cell **prints the shape and the raw bytes of each register**.

* `alpha`: shape (1024, 1); 1 bit fits in 1 byte. Values are 0 or 1.
* `beta`: shape (1024, 2); 9 bits need 2 bytes. Values are `[0 0]` (all zeros) or `[1 255]`
  (all nine bits set: $1 \cdot 256 + 255 = 511 = 2^9 - 1$).
* Because of the GHZ correlation, whenever `alpha` is 1, `beta` is all ones, and vice versa.

In [14]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

The shape of register `alpha` is (1024, 1).
The bytes in register `alpha`, shot by shot:
[[0]
 [1]
 [0]
 ...
 [1]
 [0]
 [0]]

The shape of register `beta` is (1024, 2).
The bytes in register `beta`, shot by shot:
[[  0   0]
 [  1 255]
 [  0   0]
 ...
 [  1 255]
 [  0   0]
 [  0   0]]



This cell tries to **post-select**: keep only the shots in which `alpha` was 1, and look at what
`beta` was in those shots.

* `mask` is a list of `True`/`False` values, one per shot ("did this shot pass the test?").
* `data.beta[mask[:, 0]]` keeps only the shots where the mask is `True`.

> **Look at the output:** the result has shape `(0, 2)`, meaning *no shots were kept*, although
> `alpha` was 1 in about half of them. The test `data.alpha.array == "0b1"` compares numbers (the
> stored bytes) with the *text* `"0b1"`, which is never equal. A working test compares with the
> number 1: `mask = data.alpha.array == 1`. Try it! You should then keep about 512 shots, all with
> `beta` = all ones.

In [15]:
# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == "0b1"
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

The shape of `beta` after post-selection is (0, 2).
The bytes in `beta` after post-selection:
[]


This cell **keeps only some of the bits** of every shot: `slice_bits([0, 1, 2])` keeps bits 0, 1
and 2 of `beta`. Now each shot has 3 bits (fits in 1 byte), so the shape becomes (1024, 1), with
values 0 (`000`) or 7 (`111`).

In [16]:
# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

The shape of `beta` after bit-wise slicing is (1024, 1).
The bytes in `beta` after bit-wise slicing:
[[0]
 [7]
 [0]
 ...
 [7]
 [0]
 [0]]



This cell **keeps only some of the shots**: `slice_shots([0, 1, 2, 3, 4])` keeps the first five
shots (all bits). The shape becomes (5, 2). This is handy for looking at a small sample of the data.

In [17]:
# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  0   0]
 [  1 255]
 [  0   0]
 [  1 255]
 [  1 255]]



This cell **computes expectation values directly from the measured bits**, without running the
circuit again.

* `ZZZZZZZZZ` multiplies the $Z$ results of all 9 `beta` bits. Each bit gives $+1$ (for 0) or
  $-1$ (for 1). All zeros → $+1$; all nine ones → $(-1)^9 = -1$. Each happens about half the time,
  so the average is **close to 0**.
* `IIIIIIIIZ` looks at one bit only: $+1$ or $-1$ about equally often, so it is also **close to 0**.
* The small leftover (about 0.04) comes from the random number of shots of each kind.

In [18]:
# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: 0.005859375
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: 0.005859375


This cell **joins the two registers back together**: `BitArray.concatenate_bits` puts the bits of
`alpha` and `beta` side by side, giving 10 bits per shot again (shape (1024, 2)), the same as
measuring into a single register.

In [19]:
# concatenate the bitstrings in `alpha` and `beta` to "merge" the results
# of the two registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")


The shape of the merged results is (1024, 2).
The bytes of the merged results:
[[  0   0]
 [  3 255]
 [  0   0]
 ...
 [  3 255]
 [  0   0]
 [  0   0]]



## Result metadata
In addition to the execution results, the PrimitiveResult and PubResult objects contain an optional metadata attribute about the job that was submitted. The metadata returned (if any) is implementation-specific.


This cell **prints the metadata**: extra information about the job, stored next to the results.

* `result.metadata` describes the whole job (e.g. the version of the result format).
* `result[0].metadata` describes our circuit's run, e.g. the number of shots and the circuit's
  own metadata.

What appears here depends on which Sampler you use; real IBM hardware reports more details.

In [20]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'version' : 2,

The metadata of the PubResult result is:
'shots' : 1024,
'circuit_metadata' : {},
